# import data

In [23]:
# import csv as dataframe
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os  
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [24]:
# import main expanded dataset
import pandas as pd

# Google Sheets URL
url = "https://docs.google.com/spreadsheets/d/1kgc-vj6G1Ch2IPbJng5W54lpEq-SeXeh/export?format=csv"

# 2. Let pandas read the sheet URL directly from the cloud into memory
df_raw = pd.read_csv(url)

rows = []

# Iterate through the rows of the sheet and extract the features
for i, row in df_raw.iterrows():
    # .get() in case there are missing data cells
    rows.append({ 
        "report_id": row.get("report_id", i), # fallback to loop index if column missing
        "text": row.get("text"),
        "former": row.get("former")
    })

# Construct the final training DataFrame
data = pd.DataFrame(rows)

# Output validation metrics
print(f"Total Unique Reports: {data['report_id'].nunique()}")
print(f"Matrix Dimension Profile: {data.shape}")
print("\nFirst 5 Rows Preview:")
print(data.head())

Total Unique Reports: 254
Matrix Dimension Profile: (256, 3)

First 5 Rows Preview:
   report_id                                               text  former
0       5333  PA and lateral view of the chest demonstrates ...       0
1      23171  Right arm PICC terminates in the right atrium ...       0
2      23172  Right arm PICC terminates in the right atrium ...       0
3      27573  Sequence of portable chest films dated 7 / 23 ...       0
4      27574  Sequence of portable chest films dated 4 - 22 ...       0


In [25]:
data.head(10)

,report_id,text,former
0,5333,PA and lateral view of the chest demonstrates ...,0
1,23171,Right arm PICC terminates in the right atrium ...,0
2,23172,Right arm PICC terminates in the right atrium ...,0
3,27573,Sequence of portable chest films dated 7 / 23 ...,0
4,27574,Sequence of portable chest films dated 4 - 22 ...,0
5,36780,The right PICC line is unchanged from 12TH JAN...,0
6,2584,Unremarkable cardiomediastinal silhouette . No...,0
7,3772,Dual chamber cardiac pacemaker is in place wit...,0
8,5334,PA and lateral view of the chest demonstrates ...,0
9,12053,Lung fields are clear . No pneumothorax . Hear...,0


In [3]:
# Probabilistic Target Mapping ---
# Takes discrete label indicators (0 for normal, 1 for abnormal, 2 for complex/unsure) and maps them to a continuous mathematical spectrum: 0.0, 1.0, and 0.5 respectively
label_map = {
    0: 0.0,  
    2: 0.5,  
    1: 1.0   
}

# Apply the mapping to create our continuous target variable
data['target'] = data['former'].map(label_map)

data.head(10)

,report_id,text,former,target
0,5333,PA and lateral view of the chest demonstrates ...,0,0.0
1,23171,Right arm PICC terminates in the right atrium ...,0,0.0
2,23172,Right arm PICC terminates in the right atrium ...,0,0.0
3,27573,Sequence of portable chest films dated 7 / 23 ...,0,0.0
4,27574,Sequence of portable chest films dated 4 - 22 ...,0,0.0
5,36780,The right PICC line is unchanged from 12TH JAN...,0,0.0
6,2584,Unremarkable cardiomediastinal silhouette . No...,0,0.0
7,3772,Dual chamber cardiac pacemaker is in place wit...,0,0.0
8,5334,PA and lateral view of the chest demonstrates ...,0,0.0
9,12053,Lung fields are clear . No pneumothorax . Hear...,0,0.0


In [ ]:
# Stratify Train/Test Split (80/20)
# We use the discrete 'former' column for stratification to guarantee 
# that the 0.0, 0.5, and 1.0 classes are perfectly proportional in both splits.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    data['text'],          # target feature is the raw text data
    data['target'],        # probabilistic target (0.0, 0.5, 1.0)
    test_size=0.20,         # 20% for testing
    random_state=42,        # fixed seed for reproducibility
    stratify=data['former'] # stratify by the original discrete labels to maintain class proportions
)

# Verify the Distribution Proportions ---
print("Training Set Target Proportions")
print(y_train.value_counts(normalize=True)) # Show the proportions of 0.0, 0.5, and 1.0 in the training set
print(f"Total Training Samples: {len(y_train)}\n") # Show the total number of samples in the training set

print("Testing Set Target Proportions") # Show the proportions of 0.0, 0.5, and 1.0 in the testing set
print(y_test.value_counts(normalize=True)) # Show the total number of samples in the testing set
print(f"Total Testing Samples: {len(y_test)}") # Show the total number of samples in the testing set
# proportions confirmed to be consistent across both sets, ensuring a representative distribution for model training and evaluation.

Training Set Target Proportions
target
1.0    0.544118
0.0    0.343137
0.5    0.112745
Name: proportion, dtype: float64
Total Training Samples: 204

Testing Set Target Proportions
target
1.0    0.538462
0.0    0.346154
0.5    0.115385
Name: proportion, dtype: float64
Total Testing Samples: 52


In [15]:
# CLINICAL TEXT VECTORIZATION (TF-IDF)
# Take the raw, unstructured medical text from radiology reports and converts it into a structured grid of a matrix that Random Forest can understand
# We use ngram_range=(1, 2) to capture single words and two-word phrases
# translate the text into numbers using TF-IDF (Term Frequency-Inverse Document Frequency)
# stop_words='english': strips out generic English filler words like (the, is, and) which  carry no diagnostic value for efficient model training
# ngram_range=(1, 2): tell models to capture both single words (unigrams) and two-word combinations (bigrams) to better understand the context in clinical text
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, max_df=0.95)

# Fit and transform the training text; transform the test text
X_train = tfidf_vectorizer.fit_transform(X_train_raw) # builds a "vocabulary dictionary" of every single unique word and bigram present, and then calculates the TF-IDF scores for the training set
X_test = tfidf_vectorizer.transform(X_test_raw) # converts the test text into numbers using only the vocabulary it learned from the training data to avoid data leakage

print("Data Dimensions")
print(f"Training feature matrix shape: {X_train.shape} (No of Patients in training set, Total number of unique clinical words and bigrams found in your text)")
print(f"Testing feature matrix shape:  {X_test.shape}")
print(f"Target training labels shape:  {y_train.shape}")

Data Dimensions
Training feature matrix shape: (204, 1485) (No of Patients in training set, Total number of unique clinical words and bigrams found in your text)
Testing feature matrix shape:  (52, 1485)
Target training labels shape:  (204,)


### # INITIALIZE AND TRAIN RANDOM FOREST REGRESSOR

In [16]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# INITIALIZE AND TRAIN RANDOM FOREST REGRESSOR
rf_model = RandomForestRegressor(n_estimators=300, random_state=42, ccp_alpha=0.0015, n_jobs=-1)
rf_model.fit(X_train, y_train)

# GENERATE PREDICTIONS
# Predict continuous scores for both train and test sets
y_train_prediction = rf_model.predict(X_train)
y_test_prediction = rf_model.predict(X_test)


### EVALUATE PROBABILISTIC PERFORMANCE USING MSE AND R2 SCORE

In [22]:
# EVALUATE PROBABILISTIC PERFORMANCE USING MSE AND R2 SCORE
# MSE measures the average squared difference between the predicted probabilities and the actual probabilities in the training set.
#  A lower MSE indicates better performance.
train_mse = mean_squared_error(y_train, y_train_prediction) 
test_mse = mean_squared_error(y_test, y_test_prediction)

# R2 Score measures the proportion of variance in the actual probabilities that is explained by the model's predictions in the training set.
#  A higher R2 Score indicates better performance.
train_r2 = r2_score(y_train, y_train_prediction)
test_r2 = r2_score(y_test, y_test_prediction)


print("Shallow Model Evaluation Metrics")
print(f"Train Mean Squared Error (mse): {train_mse:.4f}")
print(f"Test Mean Squared Error (mse):  {test_mse:.4f}")
print(f"Train R-squared (R2 Score):     {train_r2:.4f}")
print(f"Test R-squared (R2 Score):      {test_r2:.4f}\n")

# VIEW REAL VS PREDICTED EXAMPLES
# compare the actual probabilities with the predicted probabilities for a few test cases to see how well the model is performing on individual examples
comparison_df = pd.DataFrame({
    'Actual_Probability': y_test.values,
    'Predicted_Probability': y_test_prediction
}).round(3) # Round to 3 decimal places for easier readability

print("Sample Predictions Snapshot (First 10 Test Cases)")
print(comparison_df.head(10))

Shallow Model Evaluation Metrics
Train Mean Squared Error (mse): 0.0174
Test Mean Squared Error (mse):  0.0882
Train R-squared (R2 Score):     0.9180
Test R-squared (R2 Score):      0.5840

Sample Predictions Snapshot (First 10 Test Cases)
   Actual_Probability  Predicted_Probability
0                 0.0                  0.136
1                 0.5                  0.473
2                 0.0                  0.301
3                 0.5                  0.613
4                 0.0                  0.394
5                 0.5                  0.749
6                 1.0                  0.952
7                 0.0                  0.394
8                 1.0                  0.327
9                 0.0                  0.346


In [18]:
# calculating class specific performance to see how close the model gets on average to the actual probabilities for each of the three classes (0.0, 0.5, 1.0) in the test set
from sklearn.metrics import classification_report, confusion_matrix

# FUNCTION TO CONVERT REGRESSION TO DISCRETE CLASSIFICATION
def map_to_class(prediction):
    if prediction < 0.35: # anything below 0.35 is classified as Normal
        return 0  # Normal
    elif 0.35 <= prediction <= 0.75: # anything between 0.35 and 0.75 is classified as Unsure
        return 2  # Unsure
    else:
        return 1  # Abnormal # anything above 0.75 is classified as Abnormal



In [21]:
# using the map_to_class function to convert test targets and predictions back to discrete 0, 2, 1 classes
y_test_discrete = []

for val in y_test.values:
    mapped_value = map_to_class(val)
    y_test_discrete.append(mapped_value)

y_test_prediction_discrete = []

for val in y_test_prediction:
    mapped_value = map_to_class(val)
    y_test_prediction_discrete.append(mapped_value)

# CALCULATE DETAILED CLINICAL METRICS USING CLASSIFICATION REPORT FROM SKLEARN
print("Class-Specific Performance Breakdown")
# Mapping names back to match clinical definitions
target_names = ['Normal', 'Abnormal', 'Unsure']
# We sort the labels to match the numerical output order: 0, 1, 2
print(classification_report(y_test_discrete, y_test_prediction_discrete, 
                            labels=[0, 1, 2], # Ensure the order of labels matches the target_names
                            target_names=target_names)) # replace the default class names with the more clinically meaningful ones that correspond to the original probabilities

# CONFUSION MATRIX USING CONFUSION MATRIX FROM SKLEARN
print("Confusion Matrix")
cm = confusion_matrix(y_test_discrete, y_test_prediction_discrete, labels=[0, 1, 2])
cm_df = pd.DataFrame(cm, index=target_names, columns=['Predicted Normal', 'Predicted Abnormal', 'Predicted Unsure'])
print(cm_df)

Class-Specific Performance Breakdown
              precision    recall  f1-score   support

      Normal       0.82      0.78      0.80        18
    Abnormal       0.95      0.75      0.84        28
      Unsure       0.38      0.83      0.53         6

    accuracy                           0.77        52
   macro avg       0.72      0.79      0.72        52
weighted avg       0.84      0.77      0.79        52

Confusion Matrix
          Predicted Normal  Predicted Abnormal  Predicted Unsure
Normal                  14                   0                 4
Abnormal                 3                  21                 4
Unsure                   0                   1                 5


# FINISHED SHALLOW TRAINING

## Exploring use in labelling and possible expanding the dataset through exclusion

In [19]:
# create dataframe report id properly indexed to keep track of patients across different files
import gdown
import json

url = "https://drive.google.com/uc?id=15un_NsM_5vVV2hwegVtVu3V-j-SclXhP"
output = "data.json"

gdown.download(url, output, quiet=False)

with open(output, "r", encoding="utf-8") as f:
    data_index = json.load(f)

rows = []

for i, item in enumerate(data_index): # item is like {"0": {payload}} or {"1": {payload}}
    # item is like {"0": {payload}}
    payload = next(iter(item.values())) # get the payload dict from the item
    rows.append({ 
        "report_id": i,  # use the index as report_id to ensure uniqueness across files
        "text": payload.get("text"),
        "entities": payload.get("entities"),
        "data_split": payload.get("data_split"),
        "data_source": payload.get("data_source"),
        "num_entities": len(payload.get("entities", {})),
    })

df_index_complete = pd.DataFrame(rows)
df_index_complete["report_id"].nunique(), df_index_complete.head()
df_index_complete.shape 
df_index_complete.head()

Downloading...
From (original): https://drive.google.com/uc?id=15un_NsM_5vVV2hwegVtVu3V-j-SclXhP
From (redirected): https://drive.google.com/uc?id=15un_NsM_5vVV2hwegVtVu3V-j-SclXhP&confirm=t&uuid=378d3b4e-942d-48b5-a699-50da3763be2a
To: /Users/strangemax/syncthing/cirfolder/DATASCIENCE FOR HEALTH/Semester 2 /DASC512-202526/Assignment2/git12/512_2/data.json
100%|██████████| 237M/237M [00:19<00:00, 12.3MB/s] 


,report_id,text,entities,data_split,data_source,num_entities
0,0,Unchanged position of the left upper extremity...,"{'1': {'tokens': 'Unchanged', 'label': 'Observ...",inference,None,30
1,1,Unchanged position of the left upper extremity...,"{'1': {'tokens': 'Unchanged', 'label': 'Observ...",inference,None,30
2,2,There is redemonstration of right internal jug...,"{'1': {'tokens': 'central venous', 'label': 'A...",inference,None,26
3,3,Persistent small bilateral pleural effusions ....,"{'1': {'tokens': 'small', 'label': 'Observatio...",inference,None,28
4,4,Persistent small bilateral pleural effusions ....,"{'1': {'tokens': 'small', 'label': 'Observatio...",inference,None,28


In [44]:
# drop data data_split and data_source, num_entities columns since they are not needed for training
df_index_complete_clean = df_index_complete.drop(columns=["data_split", "data_source", "num_entities"])
df_index_complete_clean.head()

,report_id,text,entities
0,0,Unchanged position of the left upper extremity...,"{'1': {'tokens': 'Unchanged', 'label': 'Observ..."
1,1,Unchanged position of the left upper extremity...,"{'1': {'tokens': 'Unchanged', 'label': 'Observ..."
2,2,There is redemonstration of right internal jug...,"{'1': {'tokens': 'central venous', 'label': 'A..."
3,3,Persistent small bilateral pleural effusions ....,"{'1': {'tokens': 'small', 'label': 'Observatio..."
4,4,Persistent small bilateral pleural effusions ....,"{'1': {'tokens': 'small', 'label': 'Observatio..."


In [48]:
df_index_complete_clean.shape


(57805, 3)

In [49]:
data.shape

(149, 6)

In [51]:
import pandas as pd
import numpy as np

print(f"Vectorizing and scoring {len(df_index_complete_clean)} reports...")

# Vectorize the entire unlabelled text pool 
X_pool = tfidf_vectorizer.transform(df_index_complete_clean['text'])

# Run the trained Random Forest Regressor over the pool
df_index_complete_clean['predicted_score'] = rf_model.predict(X_pool)


# Ensure we have a clean list/set of IDs to compare against, regardless of 'data' type
if isinstance(data, pd.DataFrame):
    existing_ids = set(data['report_id'].tolist())
elif isinstance(data, dict) and 'report_id' in data:
    existing_ids = set(data['report_id'])
else:
    # If data is a list of records or something else, extract IDs safely
    try:
        existing_ids = set([x['report_id'] for x in data if 'report_id' in x])
    except:
        existing_ids = set(data)
        
# print existing IDs for debugging
print(f"Existing IDs in 'data': {existing_ids}")



Vectorizing and scoring 57656 reports...
Existing IDs in 'data': {3074, 52232, 1033, 8, 16913, 3607, 52247, 3622, 41, 1066, 3114, 3625, 52267, 52268, 52270, 16944, 40489, 40495, 53, 16951, 46648, 2619, 1599, 40511, 52296, 16972, 57422, 38997, 1111, 1112, 2648, 16985, 613, 52327, 1128, 52331, 1644, 37996, 1646, 37997, 1650, 1656, 1657, 1658, 56443, 3710, 1664, 23171, 23172, 2183, 1176, 1177, 2202, 15004, 38565, 3240, 1209, 698, 1724, 17086, 41664, 705, 2248, 52429, 15055, 52435, 5333, 216, 3800, 54488, 3304, 1770, 52463, 17142, 17143, 17144, 17145, 1787, 38153, 38154, 3342, 5904, 1815, 51991, 17180, 1319, 297, 16682, 17197, 3374, 3380, 821, 17205, 2873, 1851, 2364, 52033, 52548, 17221, 3400, 43336, 57167, 3407, 52048, 16726, 1881, 1887, 52577, 17250, 869, 870, 874, 3435, 52076, 16766, 52609, 52610, 16772, 2439, 3464, 16784, 52119, 52631, 409, 2969, 44953, 14750, 36780, 43437, 948, 3508, 27573, 52151, 440, 27574, 3523, 16840, 3534, 39896, 34777, 2023, 1513, 52204, 52205, 1520, 3568, 5220

In [52]:
# Exclude existing IDs using the safe set
df_index_complete_clean = df_index_complete_clean[~df_index_complete_clean['report_id'].isin(existing_ids)]

# Define filters based on your decision boundaries
# calculate how far each predicted score is from the 'Unsure' threshold of 0.5.
df_index_complete_clean['distance_from_unsure'] = (df_index_complete_clean['predicted_score'] - 0.5).abs()

# Extract top candidates
normal_candidates = df_index_complete_clean.sort_values(by='predicted_score', ascending=True).head(150).copy()
normal_candidates['assigned_label'] = 0  # Tagging as Normal

# Sort by lowest distance to 0.5 to find the best 'Unsure' candidates
unsure_candidates = df_index_complete_clean.sort_values(by='distance_from_unsure', ascending=True).head(150).copy()
unsure_candidates['assigned_label'] = 2  # Tagging as Unsure

# ombine them into the requested 'expansion_pack' dataframe
# FIX: Explicitly specify subset='report_id' so Pandas ignores the unhashable 'entities' column
expansion_pack = pd.concat([normal_candidates, unsure_candidates]).drop_duplicates(subset=['report_id'])

# Clean up helper distance column before finishing
expansion_pack = expansion_pack.drop(columns=['distance_from_unsure'])

print("\nExpansion Pack Created")
print(f"Total rows extracted: {len(expansion_pack)}")
print(expansion_pack['assigned_label'].value_counts())

# Display the first few rows to confirm structure
print("\n", expansion_pack[['report_id', 'text', 'assigned_label', 'predicted_score']].head())


Expansion Pack Created
Total rows extracted: 300
assigned_label
0    150
2    150
Name: count, dtype: int64

        report_id                                               text  \
13922      13922  There is unchanged positioning of a right - si...   
13924      13924  There is unchanged positioning of a right - si...   
53506      53506  Interval placement of NGT / OGT . Interval rem...   
30642      30642  Left upper extremity PICC is unchanged in posi...   
30643      30643  Left upper extremity PICC is unchanged in posi...   

       assigned_label  predicted_score  
13922               0         0.253333  
13924               0         0.253333  
53506               0         0.260000  
30642               0         0.265000  
30643               0         0.265000  


In [53]:
# save expansion pack to csv keeping the report id for tracking patients across files
expansion_pack.to_csv('expansion_pack.csv', index=False)

## save model for later

In [30]:
import joblib
import os

# Create a directory to store assets if they doesn't exist
save_dir = 'saved_models'
os.makedirs(save_dir, exist_ok=True)

# Define file paths
model_path = os.path.join(save_dir, 'clinical_rf_regressor_150.joblib') # Save the trained Random Forest Regressor model
vectorizer_path = os.path.join(save_dir, 'clinical_tfidf_vectorizer_150_4544.joblib') # Save the TF-IDF vectorizer to ensure new text data can be transformed in the same way

# Save both artifacts
joblib.dump(rf_model, model_path) # Save the trained Random Forest Regressor model
joblib.dump(tfidf_vectorizer, vectorizer_path) # Save the TF-IDF vectorizer to ensure new text data can be transformed in the same way

print("Assets Saved Successfully")
print(f"Model saved to: {model_path}")
print(f"Vectorizer saved to: {vectorizer_path}")

Assets Saved Successfully
Model saved to: saved_models/clinical_rf_regressor_150.joblib
Vectorizer saved to: saved_models/clinical_tfidf_vectorizer_150_4544.joblib


## loading and using saved model

In [ ]:
import joblib
import pandas as pd

# Load assets
loaded_rf_model = joblib.load('saved_models/clinical_rf_regressor.joblib')
loaded_tfidf = joblib.load('saved_models/clinical_tfidf_vectorizer.joblib')
print("Model and Vectorizer loaded successfully!")

# Load new, unseen reports for prediction
# create new dataframe with same structure
new_reports = {
    'text': [
        "xxxxxxxxxxx.",
        "xxxxxxxxxxxxxx."
    ]
}
new_df = pd.DataFrame(new_reports)

# Process and Predict
# Transform the text using the loaded vectorizer's existing vocabulary
test_transformed = loaded_tfidf.transform(new_df['text'])

# Run the probabilistic prediction
new_df['predicted_probability'] = loaded_rf_model.predict(test_transformed)

print("\nPredictions on New Data")
print(new_df)